# The Schelling Segregation Model

> **How to run:** from the repo root, `uv run jupyter lab` then open this file.

A hands-on introduction to agent-based modelling: how mild individual preferences produce strong collective segregation.

## 1. Introduction

In 1971, the economist Thomas Schelling described a striking result: even if every individual is willing to live in a mixed neighbourhood — they just prefer *not* to be a tiny minority — the collective outcome is near-total segregation. No central planner, no overt discrimination; the pattern emerges purely from local movement decisions.

The model is simple. Agents of two groups (A and B) are placed on a grid. At each time step, every agent checks what fraction of its eight immediate neighbours share its type. If that fraction is below a **tolerance threshold τ**, the agent is *unsatisfied* and moves to a random empty cell. Satisfied agents stay put. After enough steps, clusters form and segregation stabilises.

This notebook walks through the model step by step: we will build intuition for the rules, watch segregation emerge in real time, and explore how τ controls the final outcome.

## 2. Setup

We load the default configuration and set up a reproducible random key.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve() / "src"))

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import jax
import jax.numpy as jnp

from abm_geometry.config import load_config
from abm_geometry.rng import make_key
from abm_geometry.schelling import init_world, one_step, simulate
from abm_geometry.statistics.segregation import dissimilarity_index
from abm_geometry.viz.grids import plot_grid

plt.style.use("seaborn-v0_8-whitegrid")
%matplotlib inline

cfg = load_config("../experiments/configs/default.yaml")
key = make_key(cfg.seed)
print(f"Grid: {cfg.H}\u00d7{cfg.W}  |  Steps T: {cfg.T}  |  Tolerance \u03c4: {cfg.tau}  |  Density: {cfg.density}")

## 3. The Grid

The world is a **30×30 grid** where each cell is one of:
- **Empty** (white) — no agent present
- **Group A** (red) — an agent of type A
- **Group B** (blue) — an agent of type B

Initially, agents are placed at random. With `density = 0.8`, 80% of cells are occupied. Half are group A, half group B. There is no structure yet — this is the baseline before any movement.

In [ ]:
state = init_world(key, cfg)

fig, ax = plt.subplots(figsize=(5, 5))
plot_grid(state.soft_occupancy, title=f"Initial random placement  (D = {float(dissimilarity_index(state.soft_occupancy)):.3f})", ax=ax)
legend = [
    mpatches.Patch(color="#CC3333", label="Group A"),
    mpatches.Patch(color="#3333CC", label="Group B"),
    mpatches.Patch(facecolor="#F0F0F0", edgecolor="lightgrey", label="Empty"),
]
ax.legend(handles=legend, loc="lower right", fontsize=9, framealpha=0.85)
plt.tight_layout()
plt.show()

## 4. One Step: Satisfaction and Movement

At each time step the model does the following for every agent:

1. Count the agent's **8 Moore-neighbourhood** cells (the cells directly and diagonally adjacent).
2. Compute the fraction of those occupied cells that share the agent's type.
3. If that fraction is **≥ τ**, the agent is *satisfied* — it stays.
4. If it is **< τ**, the agent is *unsatisfied* — it relocates to a randomly chosen empty cell.

Below we apply a single step and highlight the cells that changed (red border).

In [ ]:
k1, _ = jax.random.split(key)
after_one = one_step(state, k1, cfg)

changed = np.array(jnp.abs(after_one.soft_occupancy - state.soft_occupancy).sum(axis=-1) > 0.1)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
plot_grid(state.soft_occupancy, title="Before (step 0)", ax=axes[0])
plot_grid(after_one.soft_occupancy, title="After (step 1)", ax=axes[1])

for r, c in zip(*np.where(changed)):
    axes[1].add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1,
                                     fill=False, edgecolor="crimson", linewidth=1.2))
plt.tight_layout()
plt.show()
print(f"Cells that moved: {int(changed.sum())}")

## 5. Full Simulation: Segregation Emerges

Let's run all T = 50 steps and compare the initial and final states.

We measure segregation with the **dissimilarity index** D:
$$D = \frac{1}{2} \sum_i \left|\frac{a_i}{A} - \frac{b_i}{B}\right|$$
where $a_i$ and $b_i$ are the type-A and type-B mass in cell $i$, and $A$, $B$ are the totals. D = 0 means perfect integration; D = 1 means complete segregation.

In [ ]:
_sim = jax.jit(simulate, static_argnums=(2,))
final_state = _sim(key, state, cfg)

d_init  = float(dissimilarity_index(state.soft_occupancy))
d_final = float(dissimilarity_index(final_state.soft_occupancy))

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
plot_grid(state.soft_occupancy,       title=f"Initial   D = {d_init:.3f}",         ax=axes[0])
plot_grid(final_state.soft_occupancy, title=f"After {cfg.T} steps   D = {d_final:.3f}", ax=axes[1])
plt.tight_layout()
plt.show()
print(f"Dissimilarity: {d_init:.3f} \u2192 {d_final:.3f}  (+{d_final - d_init:.3f})")

## 6. Segregation Over Time

How quickly does D rise, and when does it stabilise? We track the dissimilarity index at every step.

In [ ]:
d_curve = [float(dissimilarity_index(state.soft_occupancy))]
s = state
step_keys = jax.random.split(key, cfg.T)
for k in step_keys:
    s = one_step(s, k, cfg)
    d_curve.append(float(dissimilarity_index(s.soft_occupancy)))

# First t >= 10 where |D(t) - D(t-5)| < 0.01; fallback to step 35
stabilise_t = 35
for t in range(10, cfg.T):
    if abs(d_curve[t] - d_curve[t - 5]) < 0.01:
        stabilise_t = t
        break

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(cfg.T + 1), d_curve, color="#2b6cb0", linewidth=2)
ax.axvline(1, color="grey", linestyle="--", alpha=0.6)
ax.text(1.5, d_curve[1] - 0.04, "agents start\nmoving", fontsize=9, color="grey", va="top")
ax.axvline(stabilise_t, color="grey", linestyle="--", alpha=0.6)
ax.text(stabilise_t + 0.5, d_curve[stabilise_t] + 0.01,
        "segregation\nstabilises", fontsize=9, color="grey")
ax.set_xlabel("Step")
ax.set_ylabel("Dissimilarity index $D$")
ax.set_ylim(0, 1)
ax.set_title("Segregation grows rapidly then stabilises")
plt.tight_layout()
plt.show()

## 7. The Role of Tolerance

The key parameter is τ. A lower τ means agents are **more demanding**: they need a higher fraction of like-minded neighbours to be satisfied. Below we run the same simulation from the same starting state with τ ∈ {0.2, 0.4, 0.6}.

Notice how even a mild tolerance (τ = 0.4 means you are happy as long as 40% of neighbours share your type) still produces clear segregation — the core Schelling result.

In [ ]:
taus = [0.2, 0.4, 0.6]
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, tau_val in zip(axes, taus):
    s_tau = state.replace(tolerances=jnp.full((cfg.H, cfg.W), tau_val))
    final_tau = _sim(key, s_tau, cfg)
    d_tau = float(dissimilarity_index(final_tau.soft_occupancy))
    plot_grid(final_tau.soft_occupancy, title=f"\u03c4 = {tau_val}\nD = {d_tau:.3f}", ax=ax)

plt.suptitle(f"Final state after {cfg.T} steps — lower \u03c4 means more segregation",
             y=1.02, fontsize=13)
plt.tight_layout()
plt.show()